# 1 load package

In [1]:
import json
from collections import namedtuple, Counter

import numpy as np
import scanpy as sc
import pandas as pd
import squidpy as sq

## Image manipulation and geometry
from tifffile import imread
from skimage.io import imread as skimread

## Plotting imports
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, Colormap
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib import rc_context

D:\anaconda\envs\xenium-publication-env\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
D:\anaconda\envs\xenium-publication-env\lib\site-packages\numba\core\decorators.py:282: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [2]:
### Functions for registration of Xenium to Visium data and associated analyis and visualization
## Visualization functions
from companion_functions import (
    hexlist_to_cmap,
    polygons_coords_to_patch_collection,
    plot_polygons,
    hex_corner_offset,
    polygon_corners,
    celltypes,
    celltypes,
    hex_codes,
    ctype_to_code_map,
    ctype_hex_map,
    ctype_cmap,
)

## Analysis functions
from companion_functions import (
    unique_encode,
    get_xenium_to_morphology_transform_from_xenium_morphology_image,
    get_xenium_capture_polygon_um,
    transform_coordinates,
    get_median_spot_to_spot_distance_from_centroids,
    generate_space_filling_visium_polygons,
    get_visium_capture_polygon,
    __OUTSIDE_VISIUM_CAPTURE_AREA_BARCODE__,
    bin_xenium_data_to_visium_spots,
    generate_anndata_matrix_from_transcript_assignments,
)

# 2 read in data

In [ ]:
import os
import re
import glob
import pandas as pd
import anndata as ad
from scipy.sparse import csr_matrix
from concurrent.futures import ThreadPoolExecutor 

input_dir = "result download from website/counts"
output_dir = "processed_h5ad"
os.makedirs(output_dir, exist_ok=True)

def parse_cell_identifier(cell_id):
   
    match = re.match(r"^([A-Za-z]+)(\d+)_(\d+)_(\d+)$", cell_id)
    return {
        'full_id': cell_id,
        'cell_type': match.group(1) if match else "Unknown",
        'cell_number': match.group(2) if match else -1,
        'coordinate_x': match.group(3) if match else -1,
        'expression': match.group(4) if match else -1
    }

def process_single_file(input_path, output_dir):
    try:
      
        base_name = os.path.basename(input_path)
        sample_id = re.search(r"adult1_(.*?)_DAPI", base_name).group(1)
        formatted_id = sample_id.replace("-", "_") 

   
        raw_df = pd.read_csv(input_path, sep='\t', index_col=0, engine='c')
        exp_matrix = raw_df.T.astype(pd.SparseDtype("float32", 0)) 

   
        cell_metadata = pd.DataFrame(
            [parse_cell_identifier(cell_id) for cell_id in exp_matrix.index],
            index=exp_matrix.index
        )
        cell_metadata["sample_id"] = formatted_id  

      
        adata = ad.AnnData(
            X=csr_matrix(exp_matrix.sparse.to_coo()),
            obs=cell_metadata,
            var=pd.DataFrame(index=raw_df.index),
            dtype='float32'
        )
        adata.uns["sample_id"] = formatted_id 

     
        output_path = os.path.join(output_dir, f"{formatted_id}_processed.h5ad")
        adata.write(output_path, compression="gzip")  
        
        print(f" {formatted_id} -> {output_path}")
        
    except Exception as e:
        print(f"{os.path.basename(input_path)}: {str(e)}")


file_pattern = "33422-2363-adult1_*_DAPI__7.5_mu_dilated_segmentation_counts.txt"
input_files = glob.glob(os.path.join(input_dir, file_pattern))


with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(process_single_file, input_files, [output_dir]*len(input_files))

In [ ]:
import os
import glob


h5ad_files = glob.glob(os.path.join("processed_h5ad", "*_processed.h5ad"))
print("")
for f in h5ad_files:
    print(f"- {os.path.basename(f)}")

In [4]:
import anndata as ad


adatas = []
for file_path in h5ad_files:
  
    sample_id = os.path.basename(file_path).split("_processed")[0]
    
 
    adata = ad.read_h5ad(file_path)
    

    adata.obs["sample_id"] = sample_id
    
    adatas.append(adata)

In [ ]:
combined_adata = ad.concat(
    adatas,
    axis=0,          
    join="inner",    
    label="sample",  
    keys=[os.path.basename(f).split("_processed")[0] for f in h5ad_files] 
)

print("")
print(combined_adata)

In [104]:
adata.write("combined_data_all_roi_new_processed.h5ad", compression="gzip")

In [7]:
adata=combined_adata

In [ ]:
# filter cell without enough trascript
sc.pp.filter_cells(adata, min_counts=5)
sc.pp.filter_genes(adata, min_cells=3)

# 3 process data

In [ ]:
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata,resolution = 0.4)

In [28]:
#set color
custom_palette = {
    "0": "#515151",
    "1": "#AB76AE",
    "2": "#ffa5aa",
    "3": "#FE664D",
    "4": "#ff002a",
    "5": "#00FFFF",
    "6": "#32a852",
    "7": "#e5e022",
    "8": "#565DFD",
    "9": "#4fa9ff",
    "10": "#F032E6",
    "11": "#32a88e",
    "12": "#FF1493",
    "13": "#FF1493",
    "14": "#9f50f9",
    "15": "#AB76AE",
    "16": "#999999",
    "17": "#515151",
    "18": "#FFD700",
    "19": "#00FFFF",
    "20": "#3694a8",
    "21": "#FF6347",
    "22": "#ffa5aa",
    "23": "#32a852",
    "24": "#a8329e",
    "25": "#3255a8",
    "26": "#a88232",
    "27": "#32a88e",
    "28": "#a83250",
}

In [ ]:
with rc_context({'figure.figsize': (5, 5)}):
     sc.pl.umap(combined_adata,color=[ "leiden",], size=10,palette=custom_palette,legend_loc="on data", save='umap_all.pdf')

In [ ]:
sample_to_donor_map = {
    'A1_3': 'FTM120',
    'A1_4': 'FTM120',
    'A2_1': 'FTM32',
    'A2_2': 'FTM32',
    'A2_3': 'FTM32',
    'A2_4': 'FTM32',
    'B2_1': 'FTM37',
    'B2_3': 'FTM37',
    'B2_4': 'FTM37',
    'C1_1': 'FTM31',
    'C2_1': 'FTM36',
    'C2_2': 'FTM36',
    'D1_1': 'FTM31',
    'D1_2': 'FTM31',
    'D1_3': 'FTM31',
    'D2_1': 'FTM44'
}
combined_adata.obs['Donors'] = combined_adata.obs['sample'].map(sample_to_donor_map)

In [ ]:
with rc_context({'figure.figsize': (5, 5)}):
     sc.pl.umap(combined_adata,color=[ "Donors",], size=1,legend_loc="on data", save='umap_donors.png')

In [ ]:
meta_data_resolve=combined_adata.obs
meta_data_resolve.to_excel("meta_data_resolve.xlsx", index=True)

In [ ]:
sc.tl.rank_genes_groups(combined_adata, "leiden", method="wilcoxon",pts=True)

In [63]:
rank_genes_df = sc.get.rank_genes_groups_df(adata, group=None)

In [ ]:
filtered_genes_df = rank_genes_df[
    (rank_genes_df['pvals_adj'] < 0.05) & 
    (rank_genes_df['pct_nz_group'] > 0.6)  
]

In [ ]:
filtered_genes_df.to_excel("DEG_resolve_filter.xlsx", index=False)

In [64]:
top_genes = []
groups = adata.obs['leiden'].cat.categories

for group in groups:
    top_genes.append(
        sc.get.rank_genes_groups_df(adata, group).head(5)
    )

top_genes_df = pd.concat(top_genes)
gene_names = top_genes_df['names'].unique()

In [ ]:
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.dendrogram(adata, "leiden")

In [ ]:
adata.layers["scaled"] = sc.pp.scale(adata, copy=True).X

In [ ]:
sc.pl.matrixplot(
    adata,
    gene_names,
    "leiden",
    colorbar_title="mean expression",
    dendrogram=True,
    layer="scaled",
    vmin=-2,
    vmax=2,
    cmap="RdBu_r",
    save='heatmap_all_roi_new_selected_zscore_res0.3.png'
)

# 4 mapping for cell cluster

In [ ]:
adata.obs["cell_id"]=adata.obs["full_id"]

In [ ]:
import os
import re
import glob
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from shapely.geometry import Polygon

def process_sample(adata, csv_path, output_dir="mapping"):
 
    try:
       
        filename = os.path.basename(csv_path)
        
      
        match = re.search(r'([A-Z]\d+-\d+)_DAPI', filename)
        if not match:
            raise ValueError(f" {filename}")
        sample_id = match.group(1) 
        
       
        df = pd.read_csv(csv_path, header=None, names=["cell_id", "x", "y"])
        
    
        geometries = []
        cell_ids = []
        for cell_id, group in df.groupby("cell_id"):
            coords = list(zip(group["x"], group["y"]))
            polygon = Polygon(coords)
            geometries.append(polygon)
            cell_ids.append(cell_id)
        
        xenium_cell_polygons = gpd.GeoDataFrame({'cell_id': cell_ids, 'geometry': geometries})

        xenium_cell_ids = set(xenium_cell_polygons["cell_id"].astype(str))
        adata_cell_ids = set(adata.obs["cell_id"].astype(str))
        common_cell_ids = xenium_cell_ids & adata_cell_ids
        print(f" {len(common_cell_ids)} common cell_id")

       
        valid_mask = (
            xenium_cell_polygons["cell_id"].astype(str).isin(common_cell_ids) &
            xenium_cell_polygons.geometry.is_valid &
            (xenium_cell_polygons.geometry.type == 'Polygon') &
            (xenium_cell_polygons.geometry.apply(lambda g: len(g.exterior.coords) >= 3))
        )
        valid_data = xenium_cell_polygons[valid_mask].merge(
            adata.obs[["leiden"]],
            how="left",
            left_on="cell_id",
            right_index=True
        )

        if valid_data.empty:
            raise ValueError("faild")

      
        fig = plt.figure(figsize=(13, 4))
        ax = plt.gca()
        fig.patch.set_facecolor('black')
        ax.set_facecolor('black')

       
        plot_polygons(
            [list(poly.exterior.coords) for poly in valid_data.geometry],
            ax=ax,
            array=valid_data["leiden"].map(ctype_to_code_map.get),
            edgecolor="none",
            linewidth=0,
            norm=Normalize(vmin=0, vmax=len(celltypes)),
            cmap=ctype_cmap,
        )

        
        for _, row in valid_data.iterrows():
            try:
                centroid = row.geometry.centroid
                ax.text(
                    centroid.x, centroid.y,
                    str(row["leiden"]),
                    fontsize=0.01,
                    ha='center', va='center',
                    color='white'
                )
            except Exception as e:
                print(f" {str(e)}")

       
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, f"{sample_id}_new.png")
        plt.savefig(output_path, dpi=900, bbox_inches='tight', pad_inches=0.1)
        plt.close()
        print(f" {output_path}")
        return True

    except Exception as e:
        print(f" {str(e)}")
        plt.close()
        return False


csv_files = glob.glob("result download from website/segment/33422-2363-adult1_*_DAPI__7.5_mu_dilated_segmentation_cell_polygons.csv")


success_count = 0
for idx, csv_file in enumerate(csv_files, 1):
    print(f"\n {idx}/{len(csv_files)}: {os.path.basename(csv_file)}")
    if process_sample(adata, csv_file):
        success_count += 1

print(f"\n {success_count} {len(csv_files)-success_count} ")

# 5 maping of gene

In [ ]:
import os
import re
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from shapely.geometry import Polygon
from scipy.sparse import issparse


TARGET_GENES = [
    "NR5A1", "STAR", "CYP11A1",  "CYP19A1", "HSD17B1", "HSD11B1",
    "ESR1", "ESR2", "PGR", "FSHR", "INHA", "INHBB", "BMPR1B", "BMPR2",
    "TGFBR1", "TGFBR3", "ACVR1B", "ACVR2A", "ACVR2B", "ID1", "ID3", "ID4",
    "DHH", "PTCH1", "PTCH2", "GLI1", "GLI2", "JAG1", "NOTCH1", "NOTCH2",
    "NOTCH3", "WNT6", "FZD6", "SFRP2"
]
SAMPLE_CSV_PATH = "result download from website/segment/33422-2363-adult1_B2-4_DAPI__7.5_mu_dilated_segmentation_cell_polygons.csv"
OUTPUT_DIR = "Gene_Expression_No_Legend"  
ASPECT_RATIO = (19, 17) 
DPI = 900              
CMAP = "plasma"        

missing_genes = [gene for gene in TARGET_GENES if gene not in adata.var_names]
existing_genes = [gene for gene in TARGET_GENES if gene in adata.var_names]

print(f" ({len(missing_genes)}):\n{missing_genes}\n")
print(f" ({len(existing_genes)}):\n{existing_genes}\n")


def plot_genes_without_legend(adata, csv_path, genes):
   
    try:
  
        df = pd.read_csv(csv_path, header=None, names=["cell_id", "x", "y"])
        df["cell_id"] = df["cell_id"].astype(str)  
        
     
        spatial_gdf = gpd.GeoDataFrame(
            [{'cell_id': cid, 'geometry': Polygon(zip(grp.x, grp.y))} 
             for cid, grp in df.groupby("cell_id")]
        ).set_index('cell_id')

       
        sample_cells = spatial_gdf.index.intersection(adata.obs['cell_id'].astype(str))
        adata_sample = adata[adata.obs['cell_id'].astype(str).isin(sample_cells)].copy()

        
        for gene in genes:
            
            if issparse(adata_sample.X):
                expr = pd.Series(
                    adata_sample[:, gene].X.toarray().flatten(),
                    index=adata_sample.obs['cell_id'].astype(str)
                )
            else:
                expr = pd.Series(
                    adata_sample[:, gene].X.flatten(),
                    index=adata_sample.obs['cell_id'].astype(str)
                )

            
            merged_gdf = spatial_gdf.join(expr.rename('expression'), how='inner')
            
            
            valid_gdf = merged_gdf[
                merged_gdf.geometry.is_valid & 
                (merged_gdf.geometry.type == 'Polygon') &
                (merged_gdf.geometry.apply(lambda g: len(g.exterior.coords) >= 3))
            ]
            
           
            expr_values = valid_gdf['expression']
            vmax = max(expr_values.quantile(0.99), 1e-5)  
            norm = Normalize(vmin=0, vmax=vmax)

           
            fig, ax = plt.subplots(figsize=ASPECT_RATIO, facecolor='black')
            ax.set_facecolor('black')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_aspect("equal")

            
            for cid, row in valid_gdf.iterrows():
                poly = row['geometry']
                color = plt.get_cmap(CMAP)(norm(row['expression']))
                ax.fill(*poly.exterior.xy, color=color, linewidth=0)

            
            output_path = os.path.join(OUTPUT_DIR, f"{gene}_expression.png")
            plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
            plt.close()
            print(f" {os.path.basename(output_path)}")

    except Exception as e:
        print(f" {str(e)}")
        plt.close() if 'plt' in locals() else None


if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    plot_genes_without_legend(adata, SAMPLE_CSV_PATH, existing_genes)